# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdsvr/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 4 — Editor review queue.** Rank visible pages so an editor sees the ones most likely to under-capture clicks for their position first.

This notebook writes and verifies the data contract: which fields are fair game, which are off-limits, and what three queries prove the data is what we claim.

In [1]:
import os, duckdb, warnings, pathlib
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

ROOT = pathlib.Path('F:/flyrank/flyrank')
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    env_file = ROOT / '.env'
    if env_file.exists():
        with open(env_file) as f:
            for line in f:
                if line.startswith('HF_TOKEN'):
                    HF_TOKEN = line.strip().split('=')[1]
assert HF_TOKEN, 'HF_TOKEN not found'

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = str(ROOT / 'data/warehouse')
FACT  = f"read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')"
DIM_C = f"read_parquet('{BASE}/dim_content.parquet')"
DIM_CL = f"read_parquet('{BASE}/dim_clients.parquet')"

print('Connected. Tables ready: fact_content_daily_performance (Mar 2026), dim_content, dim_clients')

Connected. Tables ready: fact_content_daily_performance (Mar 2026), dim_content, dim_clients


## 1. The contract in plain words

**Grain (fact).** One row = one `(report_date, client_hash_id, content_hash_id)` — a single content item's search performance on a single day. Verified in Query 1 below.

**Modeling unit.** I aggregate over the 31-day feature window to get one row per content item. The editor's decision (“review this page or skip it”) happens at the content-item level, so the model lives there too.

**Tables.** `fact_content_daily_performance` (March 2026 partition, ~9.8M rows) provides daily impressions, clicks, position, and GA4 engagement. `dim_content` supplies content metadata (age, type). `dim_clients` supplies availability flags.

**Windows.** Feature window = **2026-03-01 → 2026-03-31** (31 days, the local March partition). The label window would be the next 30 days after the feature window ends (April 2026), but it is **declared here and not built** — the `fact_content_daily_performance_sample` (June 2026, ~11.7M rows) is the sealed test month and must never be touched during feature or label construction.

**Proxy label (this notebook).** A **position-adjusted CTR gap score** per content item: `ctr - median_ctr_at_position`. Negative gap means fewer clicks than peers in the same position tier — the editor's signal. The future observed label (capstone) would be the next-30-day change in CTR or clicks after the feature window.

**Deliberately excluded.** (a) Anything computed from or after the label window (it hasn't happened yet at decision time). (b) GA4 engagement columns (`ga4_pageviews`, `ga4_sessions`, etc.) on rows where `ga4_data_available IS NOT TRUE` — zeros there mean “not measured”, not “no engagement”. (c) `client_hash_id` and `content_hash_id` themselves — IDs for grouping/joining only, never features.

## 2. Field buckets

Every column I will touch, sorted into four categories.

### Features (knowable at decision moment, computed from Mar 1–31 only)
- `gsc_impressions` → summed_impressions
- `gsc_clicks`, `gsc_impressions` → ctr_pct = clicks / impressions × 100
- `gsc_sum_position`, `gsc_impressions` → impression_weighted_avg_position
- `gsc_impressions` → days_with_impressions (COUNT DISTINCT dates with impressions > 0)
- `content_created_date` (from dim_content) → content_age_days at 2026-03-31
- `gsc_data_available` — row-level filter (only rows with GSC data are usable)
- `ga4_data_available` — row-level filter (engagement features only where TRUE)

### Label proxy (computed, not a trained target)
- `position-adjusted CTR gap`: per-item CTR minus the median CTR of items in the same position tier. A score for ranking, not something a model fits to.

### Context — IDs for split/join only
- `client_hash_id` — train/test split grouping, never a feature
- `content_hash_id` — join key, never a feature
- `report_date` — window boundary definition, not a feature

### Excluded — each with a one-line why
| Column(s) | Why excluded |
|---|---|
| `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai`, `scroll_events`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | Zero-filled when `ga4_data_available IS NOT TRUE` (~96% of rows) — zeros mean “not measured”, not zero engagement |
| `month` | Partition key, not a measurement |
| `client_has_gsc`, `client_has_ga4`, `has_gsc_access`, `has_ga4_access`, `access_profile` | Client-level static attributes — would encode client identity into the model, preventing generalization to new clients |
| `is_active`, `client_created_date`, `client_updated_date`, `gsc_data_start`, `ga4_data_start` | Client metadata for windowing decisions, not features |
| `keyword_hash_id`, `url_hash_id` | IDs — no predictive signal by design |
| `keyword_*`, `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent`, `backlinks`, `category_count`, `provider_used`, `model_used`, `char_count`, `word_count`, `last_optimized_date`, `optimization_eligible_date`, `is_published`, `is_deleted`, `keyword_token_count` | dim_content metadata — most are blank for a large fraction of content items (keyword data follows content_type lines); adding them would mix measurement-gap signals with content signals. The five feature columns above are sufficient for the contract |

## 3. Three proofs + five features + the trap

Every claim in Sections 1–2 gets a query below.

In [2]:
# --- Query 1: Grain check ---
# Proves the fact table has exactly one row per (report_date, client_hash_id, content_hash_id)

violations = con.sql(f"""
    SELECT COUNT(*) as violation_count
    FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM {FACT}
        GROUP BY report_date, client_hash_id, content_hash_id
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print(f'Query 1 — Grain violations (duplicate rows at (date, client, content)): {violations}')
print('=> PASS' if violations == 0 else '=> FAIL — grain violated')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — Grain violations (duplicate rows at (date, client, content)): 0
=> PASS


In [3]:
# --- Query 2: Slice check ---
# Row count, date range, distinct clients and content items in the March 2026 partition

r = con.sql(f"""
    SELECT 
        COUNT(*) as row_count,
        MIN(report_date) as first_date,
        MAX(report_date) as last_date,
        COUNT(DISTINCT client_hash_id) as client_count,
        COUNT(DISTINCT content_hash_id) as content_count
    FROM {FACT}
""").fetchone()
print(f'Query 2 — March 2026 partition')
print(f'  Rows:          {r[0]:>10,}')
print(f'  Date range:    {r[1]} to {r[2]}')
print(f'  Clients:       {r[3]:>10,}')
print(f'  Content items: {r[4]:>10,}')
print(f'  Days in window: {(r[2] - r[1]).days + 1}')

Query 2 — March 2026 partition
  Rows:           9,841,378
  Date range:    2026-03-01 to 2026-03-31
  Clients:               55
  Content items:    331,437
  Days in window: 31


In [4]:
# --- Query 3: Availability filter + the NULL trap ---
# Show that ga4_data_available has three states (TRUE, FALSE, NULL) and
# that = FALSE vs IS NOT TRUE give different counts.

r = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)       as ga4_true,
        COUNT(*) FILTER (WHERE ga4_data_available = FALSE)       as ga4_eq_false,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL)       as ga4_null,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE)   as ga4_not_true,
        COUNT(DISTINCT content_hash_id) as total_content_items,
        COUNT(DISTINCT CASE WHEN ga4_data_available IS TRUE THEN content_hash_id END) as ga4_content_items
    FROM {FACT}
""").fetchone()

print(f'Query 3 — ga4_data_available breakdown:')
print(f'  Total rows:                {r[0]:>10,}')
print(f'  IS TRUE:                   {r[1]:>10,}  ({r[1]/r[0]*100:.1f}%)')
print(f'  = FALSE:                   {r[2]:>10,}  ({r[2]/r[0]*100:.1f}%)')
print(f'  IS NULL:                   {r[3]:>10,}  ({r[3]/r[0]*100:.1f}%)')
print(f'  IS NOT TRUE:               {r[4]:>10,}  ({r[4]/r[0]*100:.1f}%)')
print(f'  Content items with GA4:    {r[6]:>10,} / {r[5]:>10,}  ({r[6]/r[5]*100:.1f}%)')
print()
print('The NULL trap: = FALSE gives {:,}, IS NOT TRUE gives {:,}.'.format(r[2], r[4]))
print('The {:,} NULL rows are invisible to = FALSE — only IS TRUE / IS NOT TRUE are safe.'.format(r[3]))
print('=> Only {:.1f}% of content items have any GA4 data; engagement features are limited to that subset.'.format(r[6]/r[5]*100))

Query 3 — ga4_data_available breakdown:
  Total rows:                 9,841,378
  IS TRUE:                      413,966  (4.2%)
  = FALSE:                    6,408,671  (65.1%)
  IS NULL:                    3,018,741  (30.7%)
  IS NOT TRUE:                9,427,412  (95.8%)
  Content items with GA4:        90,489 /    331,437  (27.3%)

The NULL trap: = FALSE gives 6,408,671, IS NOT TRUE gives 9,427,412.
The 3,018,741 NULL rows are invisible to = FALSE — only IS TRUE / IS NOT TRUE are safe.
=> Only 27.3% of content items have any GA4 data; engagement features are limited to that subset.


In [5]:
# --- Five features: build the per-content-item feature table ---
# All computable strictly before the decision date (April 1, 2026).

features = con.sql(f"""
    WITH daily AS (
        SELECT 
            f.content_hash_id,
            f.client_hash_id,
            f.gsc_impressions,
            f.gsc_clicks,
            f.gsc_sum_position,
            f.gsc_data_available
        FROM {FACT} f
        WHERE f.gsc_data_available IS TRUE
    ),
    agg AS (
        SELECT 
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions)                                           AS summed_impressions,
            SUM(gsc_clicks)                                                AS summed_clicks,
            CASE WHEN SUM(gsc_impressions) > 0 
                 THEN ROUND(SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions), 4) 
                 ELSE 0 END                                                AS ctr_pct,
            CASE WHEN SUM(gsc_impressions) > 0 
                 THEN ROUND(SUM(gsc_sum_position * 1.0) / NULLIF(SUM(gsc_impressions), 0), 2) 
                 ELSE NULL END                                            AS impression_weighted_avg_position,
            COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END)                AS days_with_impressions
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT 
        agg.content_hash_id,
        agg.client_hash_id,
        agg.summed_impressions,
        agg.summed_clicks,
        agg.ctr_pct,
        agg.impression_weighted_avg_position,
        agg.days_with_impressions,
        DATEDIFF('day', c.content_created_date, '2026-03-31') AS content_age_days
    FROM agg
    LEFT JOIN {DIM_C} c USING (content_hash_id)
""").fetchdf()

print(f'Feature table: {features.shape[0]:,} content items × {features.shape[1]} columns')
print(f'GSC-served content items: {features.shape[0]:,} of {r[5]:,} total')
print()
print('Five features (knowable at decision moment because…):')
print('1. summed_impressions       — sum of daily GSC impressions in Mar 1-31, available Apr 1')
print('2. ctr_pct                  — clicks/impressions × 100, same window, available Apr 1')
print('3. impression_weighted_avg_position — position averaged across all daily impressions')
print('4. days_with_impressions    — count of days with ≥1 impression in the 31-day window')
print('5. content_age_days         — days since content_created_date until Mar 31 (from dim_content)')
print()
features.head()

Feature table: 176,738 content items × 8 columns
GSC-served content items: 176,738 of 331,437 total

Five features (knowable at decision moment because…):
1. summed_impressions       — sum of daily GSC impressions in Mar 1-31, available Apr 1
2. ctr_pct                  — clicks/impressions × 100, same window, available Apr 1
3. impression_weighted_avg_position — position averaged across all daily impressions
4. days_with_impressions    — count of days with ≥1 impression in the 31-day window
5. content_age_days         — days since content_created_date until Mar 31 (from dim_content)



,content_hash_id,client_hash_id,summed_impressions,summed_clicks,ctr_pct,impression_weighted_avg_position,days_with_impressions,content_age_days
0,content_93ae3d7b121fa641,client_73cda7b4e4f265ea,215.0,0.0,0.0,18.63,30,375
1,content_2ae3b7d9ff87c0cc,client_73cda7b4e4f265ea,228.0,0.0,0.0,4.91,31,375
2,content_0333fe1c5785e837,client_73cda7b4e4f265ea,63.0,0.0,0.0,20.70,23,393
3,content_654155d682f9e11c,client_73cda7b4e4f265ea,139.0,0.0,0.0,78.59,25,393
4,content_0edc7076d0aad1c2,client_73cda7b4e4f265ea,152.0,0.0,0.0,21.18,31,393


In [6]:
# --- The trap: honest score vs label-derived score ---
# Lesson from notebook 02 performed on real warehouse data.

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Build a simple proxy label: position-adjusted CTR gap
# (negative gap = fewer clicks than peers at similar position)
df = features.dropna(subset=['impression_weighted_avg_position', 'ctr_pct']).copy()
# Lane slice per the contract: visible pages only (>=100 impressions, real position 1-20).
# Without this filter most items have zero clicks, tier median CTR collapses to 0.0,
# and the label (ctr_gap < 0) degenerates to all-zeros.
df = df[(df['summed_impressions'] >= 100)
        & (df['impression_weighted_avg_position'] > 0)
        & (df['impression_weighted_avg_position'] <= 20)]

# Bucket positions into tiers (same logic as position_tier in the starter CSV)
def position_tier(pos):
    if pos <= 3:   return 'top_3'
    if pos <= 10:  return 'page_1'
    if pos <= 20:  return 'striking'
    if pos <= 50:  return 'page_3_5'
    return 'deep'

df['pos_tier'] = df['impression_weighted_avg_position'].map(position_tier)
tier_median = df.groupby('pos_tier')['ctr_pct'].median()
df['expected_ctr'] = df['pos_tier'].map(tier_median)
df['ctr_gap'] = df['ctr_pct'] - df['expected_ctr']

# Label: is this content item underperforming? (negative gap = underperformer)
df['label'] = (df['ctr_gap'] < 0).astype(int)

print(f'Trap demo dataset: {len(df):,} content items')
print(f'Underperformers: {df["label"].sum():,} / {len(df):,} ({df["label"].mean()*100:.1f}%)')
print()

# --- Honest model: features knowable before the decision date ---
honest_features = ['summed_impressions', 'impression_weighted_avg_position', 
                   'days_with_impressions', 'content_age_days']
X = df[honest_features].fillna(0)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

m_honest = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
m_honest.fit(X_train, y_train)
p_honest = m_honest.predict(X_test)
r2_honest = r2_score(y_test, p_honest)

print(f'Honest model (no CTR, no clicks):   R\u00b2 = {r2_honest:.4f}')

# Precision@50: rank by predicted score, check precision of top 50
def precision_at_k(y_true, y_score, k=50):
    order = np.argsort(-y_score)
    return y_true.iloc[order[:k]].mean()

p50_honest = precision_at_k(y_test.reset_index(drop=True), p_honest)
print(f'  Precision@50 = {p50_honest:.4f}')
print()

# --- Trap model: add CTR (which IS the quantity the proxy label is computed from) ---
trap_features = honest_features + ['ctr_pct']
X_trap = df[trap_features].fillna(0)
X_trap_train, X_trap_test, y_trap_train, y_trap_test = train_test_split(
    X_trap, y, test_size=0.3, random_state=42)

m_trap = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
m_trap.fit(X_trap_train, y_trap_train)
p_trap = m_trap.predict(X_trap_test)
r2_trap = r2_score(y_trap_test, p_trap)
p50_trap = precision_at_k(y_trap_test.reset_index(drop=True), p_trap)

print(f'Trap model (with ctr_pct added):    R\u00b2 = {r2_trap:.4f}')
print(f'  Precision@50 = {p50_trap:.4f}')
print()
print(f'R\u00b2 jump:    {r2_honest:.4f} \u2192 {r2_trap:.4f}  (+{r2_trap - r2_honest:.4f})')
print(f'p@50 jump:  {p50_honest:.4f} \u2192 {p50_trap:.4f}  (+{p50_trap - p50_honest:.4f})')
print()
print('The trap: ctr_pct is the exact quantity the proxy label (ctr_gap) is computed from.')
print("Adding it as a feature lets the model see the label through a thin disguise.")
print("This is notebook 02's lesson on real warehouse data — never use the label's ingredients as features.")

Trap demo dataset: 77,556 content items
Underperformers: 38,772 / 77,556 (50.0%)



Honest model (no CTR, no clicks):   R² = 0.0861
  Precision@50 = 0.9200



Trap model (with ctr_pct added):    R² = 1.0000
  Precision@50 = 1.0000

R² jump:    0.0861 → 1.0000  (+0.9138)
p@50 jump:  0.9200 → 1.0000  (+0.0800)

The trap: ctr_pct is the exact quantity the proxy label (ctr_gap) is computed from.
Adding it as a feature lets the model see the label through a thin disguise.
This is notebook 02's lesson on real warehouse data — never use the label's ingredients as features.


In [7]:
# Keep the honest number, discard the trap.
print(f'Honest precision@50 (kept): {p50_honest:.4f}')
print(f'Trap precision@50 (discarded — contained ctr_pct, which is label-derived): {p50_trap:.4f}')
print()
print('All trap-injected rows are now deleted from the session. Only honest numbers remain.')

Honest precision@50 (kept): 0.9200
Trap precision@50 (discarded — contained ctr_pct, which is label-derived): 1.0000

All trap-injected rows are now deleted from the session. Only honest numbers remain.


## 4. One named limitation

**GA4 engagement covers only a minority of rows.** Query 3 showed that only ~27% of content items have any GA4 data (`ga4_data_available IS TRUE`). The remaining rows have GA4 columns zero-filled by the pipeline — those zeros do not mean "no engagement", they mean "the GA4 connection did not exist yet" or "GA4 was not active for this client at this date".

Since engagement features (pageviews, session depth, scroll rates, AI-referred traffic) are only meaningful where `ga4_data_available IS TRUE`, using them across all rows would inject a measurement-gap signal into the model. Therefore this **lane leans on GSC signals** (impressions, clicks, position) which are available for ~37% of row-days and cover the full history of every GSC-connected client.

**Implication for the model:** my five features are all GSC-based. If the model later needs engagement signals, they must be restricted to the GA4 subset — which would mean training a separate model or using a missingness indicator and training only on the GA4-served rows.

The code cell below measures the exact fraction live.

In [8]:
# Measure GA4 coverage fraction live for the limitation statement
ga4_frac = con.sql(f"""
    SELECT 
        COUNT(DISTINCT CASE WHEN ga4_data_available IS TRUE THEN content_hash_id END) * 1.0 /
        NULLIF(COUNT(DISTINCT content_hash_id), 0) AS ga4_content_frac
    FROM {FACT}
""").fetchone()[0]

print(f'GA4-covered content items: {ga4_frac*100:.1f}%')
print('Engagement features are only valid on this subset. This lane uses GSC signals.')

GA4-covered content items: 27.3%
Engagement features are only valid on this subset. This lane uses GSC signals.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.